# Chapter 7 — Search In-Depth
## Topic 2: Dense Retrieval — Recap and Its Specific Weaknesses

**Run cells in order. Each cell is a self-contained module.**

- Module 1: Setup (loads embedding model + shared knowledge base)
- Module 2: The measured weakness — reproduces Chapter 3 numbers (0.4528 vs 0.4135)
- Module 3: Brute-force dense retrieval (Chapter 3 pattern) + exact-identifier failure
- Module 4: Qdrant dense retrieval (Chapter 6 upgrade) with metadata filtering
- Module 5: BM25 vs Dense head-to-head + why scores can't be averaged (motivates RRF)

**Install:** `pip install sentence-transformers numpy qdrant-client rank-bm25`


## Module 1: Setup and Model Loading

Run this first. Loads `model` and pre-embeds `KNOWLEDGE_BASE` — shared by all later modules.

In [ ]:
"""
Chapter 7 Topic 2 — Dense Retrieval
MODULE 1: Setup and Model Loading

Run this cell first. It loads the embedding model and defines
the shared knowledge base used by all other modules.

Install: pip install sentence-transformers numpy
"""

import numpy as np
from sentence_transformers import SentenceTransformer

# -----------------------------------------------------------------------
# Constants — match these to Chapter 6's Qdrant collection config
# -----------------------------------------------------------------------
EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"  # 384-dim, multilingual
EMBED_DIM = 384

# Load once here — all subsequent modules import from this
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Model architecture:\n{model}")
print(f"Embedding dimension: {EMBED_DIM}")

# -----------------------------------------------------------------------
# Shared knowledge base — 6 policy chunks from 4 source documents
# Simulates the 17-page RAG knowledge base in your project
# -----------------------------------------------------------------------
KNOWLEDGE_BASE = [
    {
        "id": 0,
        "text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
        "source": "01_FD_FAQ.pdf",
        "doc_type": "faq",
    },
    {
        "id": 1,
        "text": "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.",
        "source": "04_FD_Policy_Reference.pdf",
        "doc_type": "policy",
    },
    {
        "id": 2,
        "text": "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum for resident individuals.",
        "source": "02_FD_Product_Guide.pdf",
        "doc_type": "product",
    },
    {
        "id": 3,
        "text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
        "source": "02_FD_Product_Guide.pdf",
        "doc_type": "product",
    },
    {
        "id": 4,
        "text": "At maturity the FD principal and interest are credited to your registered bank account within 2 working days.",
        "source": "01_FD_FAQ.pdf",
        "doc_type": "faq",
    },
    {
        "id": 5,
        "text": "Early exit from your deposit account will attract a deduction from your returns.",
        "source": "04_FD_Policy_Reference.pdf",
        "doc_type": "policy",
    },
]

# Pre-embed the knowledge base (happens once at ingest time)
kb_texts = [doc["text"] for doc in KNOWLEDGE_BASE]
kb_embeddings = model.encode(kb_texts, normalize_embeddings=True, show_progress_bar=False)
for i, doc in enumerate(KNOWLEDGE_BASE):
    doc["embedding"] = kb_embeddings[i]

print(f"\nKnowledge base: {len(KNOWLEDGE_BASE)} chunks pre-embedded")
print(f"Embedding matrix shape: {kb_embeddings.shape}")
print("\nModule 1 complete. Run Module 2 to see cosine similarity in action.")


## Module 2: The Measured Weakness — Reproducing Chapter 3 Numbers

Reproduces `similarity(FD-A, FD-B) = 0.4528` vs `similarity(FD-A, Non-FD) = 0.4135` from your own `2_Embedding_Semantic_Search.ipynb`. Gap = 0.0393.

**Requires:** Module 1

In [ ]:
"""
Chapter 7 Topic 2 — Dense Retrieval
MODULE 2: The Measured Weakness — Reproducing Chapter 3 Numbers

Reproduces the exact similarity scores from Chapter_3/2_Embedding_Semantic_Search.ipynb:
  similarity(FD-A, FD-B)   = 0.4528  ← same complaint, different words
  similarity(FD-A, Non-FD) = 0.4135  ← completely different topic
  Gap: only 0.0393 — the core evidence that dense retrieval alone is insufficient

Requires Module 1 to have been run first (model must be loaded).
"""

import numpy as np

# -----------------------------------------------------------------------
# The three emails from Chapter 3 — real emails from fd_dataset_messy.csv
# -----------------------------------------------------------------------
EMAIL_FD_A = (
    "The machurity proceeds of my FD BJ2024FD4422 were supposed to come "
    "to my bank account but nothing recieved yet. Where is my money? "
    "Thanks, Geeta Pandey"
)

EMAIL_FD_B = (
    "Hello ji, My amnt is pending from your side since June. Plz credit "
    "it fast to my bank. Thanks and regards, Meena Patel"
)

EMAIL_NON_FD = (
    "Sir ji, App me login nahi ho raha. OTP aata hi nahi. Teen din se "
    "try kar raha hoon. Kya problem hai?"
)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """General formula — works with or without normalisation.
    For normalised vectors (||a|| = ||b|| = 1), this equals np.dot(a, b)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def embed_texts(texts: list) -> np.ndarray:
    """Batch embedding — one call for all texts."""
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)


# -----------------------------------------------------------------------
# Embed all three emails in one batch call
# -----------------------------------------------------------------------
print("Embedding 3 emails...")
vecs = embed_texts([EMAIL_FD_A, EMAIL_FD_B, EMAIL_NON_FD])
vec_fd_a, vec_fd_b, vec_non_fd = vecs[0], vecs[1], vecs[2]

# -----------------------------------------------------------------------
# Reproduce the Chapter 3 numbers
# -----------------------------------------------------------------------
sim_fd_fd = cosine_similarity(vec_fd_a, vec_fd_b)
sim_fd_nonfd = cosine_similarity(vec_fd_a, vec_non_fd)
gap = sim_fd_fd - sim_fd_nonfd

print(f"\n{'='*65}")
print("CHAPTER 3 NUMBERS — REPRODUCED")
print(f"{'='*65}")
print(f"\nFD complaint A : {EMAIL_FD_A[:70]}...")
print(f"FD complaint B : {EMAIL_FD_B[:70]}...")
print(f"Non-FD email   : {EMAIL_NON_FD[:70]}...")
print()
print(f"similarity(FD-A, FD-B)   = {sim_fd_fd:.4f}   ← same complaint, different words")
print(f"similarity(FD-A, Non-FD) = {sim_fd_nonfd:.4f}   ← completely different topic")
print()
print(f"Gap = {sim_fd_fd:.4f} - {sim_fd_nonfd:.4f} = {gap:.4f}")
print()
print("Interpretation:")
print(f"  Less than {gap:.1%} separates 'same FD complaint' from 'completely different topic'")
print(f"  At 2,500 emails and top-5 retrieval, this thin margin produces frequent misretrieval")
print(f"  Dense retrieval alone is insufficient for short (~31 word) Hinglish emails")

# -----------------------------------------------------------------------
# Sanity check: dot product == cosine similarity for normalised vectors
# -----------------------------------------------------------------------
dot_product = float(np.dot(vec_fd_a, vec_fd_b))
print(f"\nSanity check (from Chapter 3):")
print(f"  cosine_similarity(FD-A, FD-B) = {sim_fd_fd:.6f}")
print(f"  np.dot(FD-A, FD-B)            = {dot_product:.6f}  (same, since vectors are normalised)")
print(f"  They match: {np.isclose(dot_product, sim_fd_fd)}")

print("\nModule 2 complete. Run Module 3 to see brute-force dense retrieval.")


## Module 3: Brute-Force Dense Retrieval (Chapter 3 Pattern)

Shows the in-memory O(n) search pattern, then the exact-identifier failure mode (FD reference numbers embedding too close to each other).

**Requires:** Module 1

In [ ]:
"""
Chapter 7 Topic 2 — Dense Retrieval
MODULE 3: Brute-Force Dense Retrieval (Chapter 3 in-memory approach)

Shows the exact search pattern from Chapter_3/2_Embedding_Semantic_Search.ipynb:
  - Embed query
  - Compare against every stored vector (O(n) — linear scan)
  - Return top-k by cosine similarity

Also demonstrates the exact-identifier failure:
  - Querying for "BJ2019FD7717" via semantic search
  - Shows why validate_fd_reference() uses exact-match lookup instead

Requires Module 1 to have been run first.
"""

import numpy as np


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def dense_search_brute_force(query: str, knowledge_base: list, top_k: int = 3) -> list:
    """Brute-force dense retrieval — the Chapter 3 in-memory pattern.
    O(n) per query: embeds the query, scores against every stored vector.
    Fine at 17 pages; becomes slow at tens of thousands of chunks."""

    # 1. Embed the query using the same model used at ingest time
    query_vec = model.encode(query, normalize_embeddings=True, show_progress_bar=False)

    # 2. Score against every stored document vector
    scored = []
    for doc in knowledge_base:
        score = cosine_similarity(query_vec, doc["embedding"])
        scored.append((score, doc))

    # 3. Sort descending by score, return top-k
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


def print_results(query: str, results: list) -> None:
    print(f"\nQuery: '{query}'")
    print(f"{'─'*65}")
    for rank, (score, doc) in enumerate(results):
        print(f"  Rank {rank+1} | Score: {score:.4f} | [{doc['doc_type'].upper()}] {doc['source']}")
        print(f"           {doc['text'][:70]}...")


# -----------------------------------------------------------------------
# Demo 1: Standard semantic query — where dense retrieval works well
# -----------------------------------------------------------------------
print(f"{'='*65}")
print("DEMO 1: Semantic query — vocabulary mismatch handled correctly")
print(f"{'='*65}")
print("Query uses different words from documents but same meaning")
print("BM25 would score 0 here; dense retrieval finds semantic match")

query_semantic = "early exit from my fixed deposit account"
results = dense_search_brute_force(query_semantic, KNOWLEDGE_BASE, top_k=3)
print_results(query_semantic, results)

print("\n  'Early exit' matched 'premature withdrawal/closure' semantically.")
print("  This is what dense retrieval solves that BM25 cannot.")

# -----------------------------------------------------------------------
# Demo 2: Hinglish query — where dense retrieval partially works
# -----------------------------------------------------------------------
print(f"\n{'='*65}")
print("DEMO 2: Hinglish query — partial semantic bridge")
print(f"{'='*65}")
print("Query in Hinglish; documents in English")
print("64.4% of your 2,500 emails are Hinglish — this is the dominant case")

query_hinglish = "mera FD band karna hai penalty kitni hogi"
results = dense_search_brute_force(query_hinglish, KNOWLEDGE_BASE, top_k=3)
print_results(query_hinglish, results)

print("\n  Scores are lower than for English queries — Hinglish-English")
print("  semantic bridge is imperfect in paraphrase-multilingual-MiniLM.")
print("  This motivates hybrid search + query expansion (Topics 4, 9).")

# -----------------------------------------------------------------------
# Demo 3: Exact identifier query — where dense retrieval fails
# -----------------------------------------------------------------------
print(f"\n{'='*65}")
print("DEMO 3: FD reference number query — dense retrieval failure mode")
print(f"{'='*65}")
print("Reference numbers like BJ2019FD7717 are exact identifiers.")
print("Dense retrieval cannot distinguish between different reference numbers.")

# Add two fake customer records to demonstrate the cross-customer issue
customer_chunks = [
    {
        "id": 10,
        "text": "Customer Shobha Chopra FD BJ2019FD7717 status: Closed Premature maturity date 2021-11-28 amount 391000",
        "source": "fd_master_database",
        "doc_type": "customer_record",
    },
    {
        "id": 11,
        "text": "Customer Ramesh Patel FD BJ2021FD4432 status: Active maturity date 2024-06-15 amount 250000",
        "source": "fd_master_database",
        "doc_type": "customer_record",
    },
    {
        "id": 12,
        "text": "Customer Anita Sharma FD BJ2022FD8891 status: Active maturity date 2025-03-20 amount 500000",
        "source": "fd_master_database",
        "doc_type": "customer_record",
    },
]

# Pre-embed the customer chunks
for doc in customer_chunks:
    doc["embedding"] = model.encode(doc["text"], normalize_embeddings=True, show_progress_bar=False)

# Query for a specific reference number
query_reference = "check FD reference BJ2019FD7717"
results = dense_search_brute_force(query_reference, customer_chunks, top_k=3)

print_results(query_reference, results)

# Compute pairwise similarities between reference number embeddings
ref_embeddings = [model.encode(doc["text"], normalize_embeddings=True, show_progress_bar=False)
                  for doc in customer_chunks]
sim_a_b = cosine_similarity(ref_embeddings[0], ref_embeddings[1])
sim_a_c = cosine_similarity(ref_embeddings[0], ref_embeddings[2])

print(f"\n  Similarity between BJ2019FD7717 and BJ2021FD4432: {sim_a_b:.4f}")
print(f"  Similarity between BJ2019FD7717 and BJ2022FD8891: {sim_a_c:.4f}")
print(f"\n  Reference numbers embed to SIMILAR vectors because they share")
print(f"  the same subword tokens (BJ, FD) — the model cannot distinguish them.")
print(f"  This is why validate_fd_reference() + get_fd_record() use exact-match")
print(f"  SQL-style lookup, NOT semantic search. Never use dense retrieval for")
print(f"  exact identifiers: account numbers, reference codes, customer IDs.")

print("\nModule 3 complete. Run Module 4 to see how Qdrant improves on this.")


## Module 4: Qdrant Dense Retrieval (Chapter 6 Upgrade)

Upserts the knowledge base into an in-memory Qdrant collection, runs filtered and unfiltered dense search, compares against the Module 3 brute-force results.

**Requires:** Module 1

In [ ]:
"""
Chapter 7 Topic 2 — Dense Retrieval
MODULE 4: Qdrant Dense Retrieval (Chapter 6 upgrade)

Shows the Chapter 6 Qdrant implementation:
  - Upsert knowledge base chunks into Qdrant collection
  - Query using client.query_points() with a dense vector
  - Add metadata filtering on top of dense search

This is the upgrade from Chapter 3's brute-force list-scan to
Chapter 6's HNSW-indexed, persistent, filterable vector store.

Requires Module 1 to have been run first.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    PayloadSchemaType,
    Filter,
    FieldCondition,
    MatchValue,
)

# -----------------------------------------------------------------------
# Setup: in-memory Qdrant client (no Docker needed)
# -----------------------------------------------------------------------
COLLECTION_NAME = "fd_knowledge_base_ch7"
VECTOR_SIZE = 384

client = QdrantClient(":memory:")

# Create collection with cosine distance
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)

# Add payload index on doc_type for filtered search
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="doc_type",
    field_schema=PayloadSchemaType.KEYWORD,
)
print(f"Created Qdrant collection '{COLLECTION_NAME}' with payload index on doc_type")


def make_point_id(source: str, chunk_id: int) -> int:
    """Deterministic ID from source + chunk_id.
    Same input always produces the same ID — upsert is idempotent."""
    key = f"{source}::{chunk_id}"
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


# -----------------------------------------------------------------------
# Upsert knowledge base into Qdrant
# -----------------------------------------------------------------------
points = [
    PointStruct(
        id=make_point_id(doc["source"], doc["id"]),
        vector=doc["embedding"].tolist(),  # numpy array → list for serialisation
        payload={
            "text": doc["text"],
            "source": doc["source"],
            "doc_type": doc["doc_type"],
            "chunk_id": doc["id"],
        },
    )
    for doc in KNOWLEDGE_BASE
]

client.upsert(collection_name=COLLECTION_NAME, points=points)
info = client.get_collection(COLLECTION_NAME)
print(f"Upserted {info.points_count} points into Qdrant")


def dense_search_qdrant(query: str, top_k: int = 3, doc_type_filter: str = None) -> None:
    """Dense retrieval via Qdrant query_points().
    Replaces the brute-force loop from Module 3 with HNSW index search.
    Optionally filters by doc_type payload field during HNSW traversal."""

    # Embed query with same model used at ingest time
    query_vector = model.encode(query, normalize_embeddings=True, show_progress_bar=False).tolist()

    # Build optional payload filter
    query_filter = None
    if doc_type_filter:
        query_filter = Filter(
            must=[FieldCondition(key="doc_type", match=MatchValue(value=doc_type_filter))]
        )

    # query_points() replaces the old client.search() removed in qdrant-client >= 1.9
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,
        limit=top_k,
        with_payload=True,
    ).points

    filter_label = f"doc_type='{doc_type_filter}'" if doc_type_filter else "no filter"
    print(f"\nQuery: '{query}' [{filter_label}]")
    print(f"{'─'*65}")
    for rank, r in enumerate(results):
        print(f"  Rank {rank+1} | Score: {r.score:.4f} | [{r.payload['doc_type'].upper()}] {r.payload['source']}")
        print(f"           {r.payload['text'][:70]}...")


# -----------------------------------------------------------------------
# Demo 1: Unfiltered dense search
# -----------------------------------------------------------------------
print(f"\n{'='*65}")
print("DEMO 1: Unfiltered dense search (all doc types)")
print(f"{'='*65}")
dense_search_qdrant("what happens when I close my FD early", top_k=3)

# -----------------------------------------------------------------------
# Demo 2: Filtered dense search — only FAQ documents
# -----------------------------------------------------------------------
print(f"\n{'='*65}")
print("DEMO 2: Filtered dense search — only FAQ documents")
print(f"{'='*65}")
print("Filter applied DURING HNSW traversal, not after fetching results.")
print("Returns k results that both match semantically AND match the filter.")
dense_search_qdrant("what happens when I close my FD early", top_k=2, doc_type_filter="faq")

# -----------------------------------------------------------------------
# Demo 3: Filtered dense search — only policy documents
# -----------------------------------------------------------------------
print(f"\n{'='*65}")
print("DEMO 3: Filtered dense search — only policy documents")
print(f"{'='*65}")
dense_search_qdrant("premature withdrawal penalty", top_k=2, doc_type_filter="policy")

# -----------------------------------------------------------------------
# Demo 4: Compare brute-force Chapter 3 vs Qdrant Chapter 6
# -----------------------------------------------------------------------
print(f"\n{'='*65}")
print("DEMO 4: Chapter 3 (brute-force) vs Chapter 6 (Qdrant HNSW)")
print(f"{'='*65}")
test_query = "early FD closure penalty"
query_vec = model.encode(test_query, normalize_embeddings=True, show_progress_bar=False)

# Chapter 3 approach: manual loop
ch3_scores = []
for doc in KNOWLEDGE_BASE:
    score = float(np.dot(query_vec, doc["embedding"]))
    ch3_scores.append((score, doc["text"][:60]))
ch3_scores.sort(reverse=True)

# Chapter 6 approach: Qdrant
ch6_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec.tolist(),
    limit=3,
    with_payload=True,
).points

print(f"\nQuery: '{test_query}'")
print("\nChapter 3 (brute-force, exact):")
for i, (score, text) in enumerate(ch3_scores[:3]):
    print(f"  Rank {i+1} | {score:.4f} | {text}...")

print("\nChapter 6 (Qdrant HNSW, approximate):")
for i, r in enumerate(ch6_results):
    print(f"  Rank {i+1} | {r.score:.4f} | {r.payload['text'][:60]}...")

print("\n  At 6 chunks, results are identical — HNSW approximation only")
print("  becomes visible at thousands of vectors. The upgrade from Chapter 3")
print("  to Chapter 6 is about: persistence, metadata filtering, and scalability.")

print("\nModule 4 complete. Run Module 5 to see the full comparison summary.")


## Module 5: BM25 vs Dense — Head-to-Head Comparison

Shows where each method wins/loses, and why BM25 and cosine scores live on different scales (motivating RRF in Topic 4).

**Requires:** Module 1 (Module 4 not required, this module rebuilds what it needs)

In [ ]:
"""
Chapter 7 Topic 2 — Dense Retrieval
MODULE 5: BM25 vs Dense Retrieval — Head-to-Head Comparison

Shows exactly where each retrieval method wins and loses on your corpus:
  - BM25: wins on exact terms, novel names, FD reference numbers
  - Dense: wins on paraphrase/vocabulary mismatch, cross-language matching
  - Neither alone is sufficient → motivates hybrid search (Topic 4)

Also shows the score scale difference: BM25 and cosine similarity
scores are on completely different scales and cannot be directly compared
— which is why RRF (Reciprocal Rank Fusion) is used to merge them.

Requires Module 1 + Module 4 (Qdrant client and collection) to have been run.
Install: pip install rank-bm25 qdrant-client sentence-transformers
"""

import numpy as np
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list:
    return text.lower().split()


# Build BM25 index over the same knowledge base
tokenized_corpus = [tokenize(doc["text"]) for doc in KNOWLEDGE_BASE]
bm25 = BM25Okapi(tokenized_corpus, k1=1.2, b=0.75)
print(f"BM25 index built over {len(KNOWLEDGE_BASE)} knowledge base chunks")


def compare_retrieval(query: str, top_k: int = 3) -> None:
    """Side-by-side: BM25 scores vs dense cosine similarity scores.
    Highlights which method ranks each document higher and why."""

    print(f"\n{'='*65}")
    print(f"Query: '{query}'")
    print(f"{'='*65}")

    # BM25 scores
    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_ranked = sorted(
        [(bm25_scores[i], i) for i in range(len(KNOWLEDGE_BASE))],
        reverse=True
    )

    # Dense cosine similarity scores
    query_vec = model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    dense_scores = [float(np.dot(query_vec, doc["embedding"])) for doc in KNOWLEDGE_BASE]
    dense_ranked = sorted(
        [(dense_scores[i], i) for i in range(len(KNOWLEDGE_BASE))],
        reverse=True
    )

    print(f"\n  {'Rank':>4} | {'BM25 Score':>10} | {'Document':>50}")
    print(f"  {'─'*4}-+-{'─'*10}-+-{'─'*50}")
    for rank, (score, idx) in enumerate(bm25_ranked[:top_k]):
        doc = KNOWLEDGE_BASE[idx]
        print(f"  {rank+1:>4} | {score:>10.4f} | {doc['text'][:50]}...")

    print(f"\n  {'Rank':>4} | {'Dense Score':>11} | {'Document':>50}")
    print(f"  {'─'*4}-+-{'─'*11}-+-{'─'*50}")
    for rank, (score, idx) in enumerate(dense_ranked[:top_k]):
        doc = KNOWLEDGE_BASE[idx]
        print(f"  {rank+1:>4} | {score:>11.4f} | {doc['text'][:50]}...")

    # Identify agreements and disagreements in top-1
    bm25_top1 = bm25_ranked[0][1]
    dense_top1 = dense_ranked[0][1]
    if bm25_top1 == dense_top1:
        print(f"\n  AGREEMENT: Both methods rank doc {bm25_top1} first")
    else:
        print(f"\n  DISAGREEMENT: BM25 ranks doc {bm25_top1} first, Dense ranks doc {dense_top1} first")
        print(f"  BM25 top-1:  '{KNOWLEDGE_BASE[bm25_top1]['text'][:60]}...'")
        print(f"  Dense top-1: '{KNOWLEDGE_BASE[dense_top1]['text'][:60]}...'")


# -----------------------------------------------------------------------
# Test case 1: English query, overlapping vocabulary
# Both methods should agree
# -----------------------------------------------------------------------
compare_retrieval("premature withdrawal penalty FD")

# -----------------------------------------------------------------------
# Test case 2: Vocabulary mismatch — where dense wins
# BM25 returns 0 for everything (no word overlap)
# Dense finds semantic match
# -----------------------------------------------------------------------
compare_retrieval("I want to exit my deposit early what fee")

# -----------------------------------------------------------------------
# Test case 3: Hinglish query — dense partially wins
# BM25 returns 0 for everything; dense provides some signal
# -----------------------------------------------------------------------
compare_retrieval("FD band karna hai penalty batao")

# -----------------------------------------------------------------------
# Test case 4: Exact term match — where BM25 wins
# Novel product name: embedding has weak signal, BM25 finds exact match
# -----------------------------------------------------------------------

# Add a novel product chunk to demonstrate
novel_chunk_text = "Smart Saver FD offers flexible premature withdrawal with zero penalty for amounts above 5 lakhs."
novel_embedding = model.encode(novel_chunk_text, normalize_embeddings=True, show_progress_bar=False)

novel_query = "Smart Saver FD penalty rules"
tokenized_novel = [tokenize(novel_chunk_text)] + tokenized_corpus
bm25_with_novel = BM25Okapi(tokenized_novel, k1=1.2, b=0.75)

print(f"\n{'='*65}")
print("Test Case 4: Novel product name — where BM25 wins over Dense")
print(f"{'='*65}")
print(f"Query: '{novel_query}'")
print(f"Novel chunk: '{novel_chunk_text[:60]}...'")

bm25_novel_scores = bm25_with_novel.get_scores(tokenize(novel_query))
print(f"\n  BM25 score for novel 'Smart Saver FD' chunk: {bm25_novel_scores[0]:.4f}")
print(f"  BM25 scores for other chunks: {[round(s,3) for s in bm25_novel_scores[1:]]}")

query_vec_novel = model.encode(novel_query, normalize_embeddings=True, show_progress_bar=False)
dense_novel = float(np.dot(query_vec_novel, novel_embedding))
dense_existing = [float(np.dot(query_vec_novel, doc["embedding"])) for doc in KNOWLEDGE_BASE]
print(f"\n  Dense score for novel 'Smart Saver FD' chunk: {dense_novel:.4f}")
print(f"  Dense scores for existing chunks (max): {max(dense_existing):.4f}")
print(f"\n  BM25 correctly ranks the novel chunk highest via exact term match.")
print(f"  Dense retrieval may not — the embedding model has no training signal")
print(f"  for 'Smart Saver FD' as a specific product name.")

# -----------------------------------------------------------------------
# Why you cannot average BM25 and dense scores directly
# -----------------------------------------------------------------------
print(f"\n{'='*65}")
print("WHY SCORES CANNOT BE DIRECTLY AVERAGED — DIFFERENT SCALES")
print(f"{'='*65}")

query_scales = "premature withdrawal penalty"
bm25_scale_scores = bm25.get_scores(tokenize(query_scales))
dense_scale_scores = [
    float(np.dot(
        model.encode(query_scales, normalize_embeddings=True, show_progress_bar=False),
        doc["embedding"]
    ))
    for doc in KNOWLEDGE_BASE
]

print(f"\nQuery: '{query_scales}'")
print(f"\n  BM25 scores range:  {min(bm25_scale_scores):.3f} to {max(bm25_scale_scores):.3f}")
print(f"  Dense scores range: {min(dense_scale_scores):.3f} to {max(dense_scale_scores):.3f}")
print(f"\n  BM25 uses raw log-probability-derived scores with no fixed scale.")
print(f"  Dense uses cosine similarity: always in [-1, +1] for normalised vectors.")
print(f"  A naïve average of 2.1 (BM25) + 0.6 (dense) = 2.7 is meaningless:")
print(f"  BM25's scale dominates entirely, making dense retrieval irrelevant.")
print(f"\n  Solution: Reciprocal Rank Fusion (RRF) — use ranks, not scores.")
print(f"  RRF score(doc) = Σ 1/(k + rank(doc)) for each retriever")
print(f"  This makes BM25 and dense contribute equally regardless of score scale.")
print(f"  Topic 4 (Hybrid Search) covers RRF in full detail.")

print(f"\n{'='*65}")
print("CHAPTER 7 TOPIC 2 SUMMARY")
print(f"{'='*65}")
print("""
Dense retrieval: embeds both query and documents into 384-dimensional
  vector space, finds nearest neighbors by cosine similarity.

Your measured weakness (from Chapter 3):
  similarity(FD-A, FD-B)   = 0.4528  ← same complaint
  similarity(FD-A, Non-FD) = 0.4135  ← different topic
  Gap = 0.0393 — dangerously thin for short Hinglish emails

Specific failure modes:
  1. Exact identifiers (BJ2019FD7717): reference numbers embed similarly
     → use exact-match lookup via validate_fd_reference() instead
  2. Novel product names (Smart Saver FD): weak training signal
     → BM25 finds these by exact term match
  3. Hinglish queries vs English documents: imperfect cross-lingual bridge
     → query expansion helps; hybrid search provides fallback

Key rules:
  ✓ Always use same model at ingest and query time
  ✓ Always normalize_embeddings=True
  ✓ Always batch-encode, never one-at-a-time
  ✗ Never semantic search for exact identifiers
  ✗ Never average BM25 + dense scores directly (use RRF)

Next: Topic 3 (DPR, ColBERT, SPLADE) → Topic 4 (Hybrid Search + RRF)
""")
